# ARGUS Text Risk Demo

Enter raw text, choose a retriever and thresholds, then run named-entity extraction plus ARGUS risk scoring. The first run may download the NER or embedding model.

## Setup

In [2]:
from pathlib import Path
import sys

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "with_argus_eyes").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not find the With_Argus_Eyes repository root from this notebook location.")
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = lambda value: value
    display = print

from with_argus_eyes.inference import (
    ArgusTextConfig,
    analyze_text,
    available_retrievers,
    highlight_entities,
    resolve_model_artifact,
)

print("Available retrievers:", ", ".join(available_retrievers()))

Available retrievers: contriever, qwen3, jina, bge-m3, reason-embed, nv-embed, gritlm, reasonir


## Environment configuration

In [3]:
import os

# Edit these before running the analysis cells.
# Use "" for CPU/default device behavior, or values such as "0", "0,1", or "6,7" for specific GPUs.
CUDA_VISIBLE_DEVICES = "6,7"

# Keep Hugging Face downloads in a predictable location. Set to "" to use your system default.
HF_CACHE_DIR = str(repo_root / "outputs" / "cache" / "huggingface")

if CUDA_VISIBLE_DEVICES:
    os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
else:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)

if HF_CACHE_DIR:
    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["HF_HUB_CACHE"] = str(Path(HF_CACHE_DIR) / "hub")
    os.environ["HF_DATASETS_CACHE"] = str(Path(HF_CACHE_DIR) / "datasets")
    os.environ["TRANSFORMERS_CACHE"] = str(Path(HF_CACHE_DIR) / "transformers")

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<default/CPU>"))
print("HF_HOME:", os.environ.get("HF_HOME", "<system default>"))

CUDA_VISIBLE_DEVICES: 6,7
HF_HOME: /mounts/Users/cisintern/zeinabtaghavi/With_Argus_Eyes/outputs/cache/huggingface


## User configuration

In [4]:
config = ArgusTextConfig(
    retriever="contriever",
    language="en",
    ner_model="dslim/bert-base-NER",
    risk_threshold=0.3,
    ner_threshold=0.5,
    order=800,
    k=50,
    text_mode="span",
    workspace_root=repo_root,
)

artifact = resolve_model_artifact(config)
print("Selected ARGUS model artifact:")
print(artifact)

Selected ARGUS model artifact:
/mounts/Users/cisintern/zeinabtaghavi/With_Argus_Eyes/outputs/12_risk_outputs/contriever_ratio_unrelevant_below_k_50_o_800_k_50_sampled_average/models/mlp_best_contriever_S3_WD_Low_seed42.joblib


## Text input

In [ ]:
text = """
St. Martin's Church in Zillis, Switzerland, is a Romanesque church best known
for its painted wooden ceiling panels dating from the 12th century. Neanderthals
inhabited Europe and Western and Central Asia during the Middle to Late Pleistocene.
""".strip()

print(text)

St. Martin's in Zillis, Switzerland, is a Romanesque best known
for its painted wooden ceiling panels dating from the 12th century. Neanderthals
inhabited Europe and Western and Central Asia during the Middle to Late Pleistocene.


## Run analysis

In [9]:
results = analyze_text(text, config)

if not results:
    print("No named entities were found with the current NER threshold.")
else:
    print(f"Scored {len(results)} entity mentions.")

ValueError: Phrase not found in text: "St. Martin ' s"

## Results table

In [ ]:
columns = ["entity", "entity_type", "ner_score", "risk_score", "above_threshold", "retriever"]

if results:
    try:
        import pandas as pd
        display(pd.DataFrame(results)[columns].sort_values("risk_score", ascending=False))
    except ImportError:
        for row in sorted(results, key=lambda item: item["risk_score"], reverse=True):
            print({key: row[key] for key in columns})
else:
    print("Nothing to display.")

,entity,entity_type,ner_score,risk_score,above_threshold,retriever
0,St. Martin ' s Church,LOC,0.990153,0.084907,False,contriever
1,Zillis,LOC,0.977817,0.084907,False,contriever
2,Switzerland,LOC,0.999736,0.084907,False,contriever
3,Romanesque,MISC,0.990743,0.084907,False,contriever
4,Neanderthal,MISC,0.868969,0.084907,False,contriever
5,Europe,LOC,0.999537,0.084907,False,contriever
6,Western,LOC,0.997865,0.084907,False,contriever
7,Central Asia,LOC,0.997917,0.084907,False,contriever
8,Middle,MISC,0.993405,0.084907,False,contriever
9,Late P,MISC,0.932711,0.084907,False,contriever


## Highlighted text

In [7]:
if results:
    display(HTML("<div style='line-height:1.8; font-size:1rem'>" + highlight_entities(text, results) + "</div>"))
else:
    print("No highlighted entities.")